# Railway Block Planning — Data Pipeline

This notebook builds the 6 core tables for the AI-powered block planning project:

1. `01_stations_master.csv` — **REAL**
2. `02_train_timetable.csv` — **REAL**
3. `03_section_traffic_derived.csv` — **DERIVED** from real data
4. `04_asset_register.csv` — **SYNTHETIC**
5. `05_maintenance_blocks.csv` — **SYNTHETIC**, keyed to real section IDs
6. `06_daily_asset_availability.csv` — **SYNTHETIC**

**Input required:** `Train_details_22122017.csv` (the raw Indian Railways timetable) in the same folder as this notebook.


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

rng = np.random.default_rng(7)
IN_FILE = "Train_details_22122017.csv"   # put the raw file next to this notebook
OUT = "."                                  # output folder for the generated CSVs


## Step 1 — Load and clean the raw timetable

The raw file has 10 corrupted rows (5 train pairs) where a stray comma inside a field shifted every column over by one — you can spot them because `Train No` shows the letter `K` instead of a number. We drop these.


In [2]:
tt = pd.read_csv(IN_FILE, dtype=str, low_memory=False)
tt.columns = [c.strip() for c in tt.columns]

before = len(tt)
tt = tt[tt["Train No"].str.match(r"^\d+$", na=False)].copy()
dropped = before - len(tt)
print(f"Dropped {dropped} corrupted rows")

tt["SEQ"] = tt["SEQ"].astype(int)
tt["Distance"] = pd.to_numeric(tt["Distance"], errors="coerce")
tt = tt.sort_values(["Train No", "SEQ"]).reset_index(drop=True)
tt.head()


Dropped 5 corrupted rows


,Train No,Train Name,SEQ,Station Code,Station Name,Arrival time,Departure Time,Distance,Source Station,Source Station Name,Destination Station,Destination Station Name
0,10103,MANDOVI EXPR,1,CSMT,CST-MUMBAI,07:10:00,07:10:00,0.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
1,10103,MANDOVI EXPR,2,DR,DADAR,07:22:00,07:25:00,8.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
2,10103,MANDOVI EXPR,3,TNA,THANE,07:46:00,07:50:00,33.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
3,10103,MANDOVI EXPR,4,PNVL,PANVEL,08:25:00,08:30:00,70.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
4,10103,MANDOVI EXPR,5,MNI,MANGAON,10:34:00,10:35:00,177.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.


## File 1 — `01_stations_master.csv`
**Question answered: Where are the stations?**

Every unique station code + name found in the timetable. This is real data — no need for a separate download, since the timetable file already lists every station it visits.


In [3]:
stations = (tt[["Station Code", "Station Name"]]
            .drop_duplicates(subset=["Station Code"])
            .rename(columns={"Station Code": "station_code", "Station Name": "station_name"})
            .sort_values("station_code")
            .reset_index(drop=True))

stations.to_csv(f"{OUT}/01_stations_master.csv", index=False)
print(stations.shape)
stations.head()


(8148, 2)


,station_code,station_name
0,AABH,AMBIKA BHAWA
1,AADR,AMB ANDAURA
2,AAG,ANGAR
3,AAH,ITEHAR
4,AAL,AMLAI


## File 2 — `02_train_timetable.csv`
**Question answered: Which train goes where, and when?**

The cleaned raw timetable, with columns renamed to a consistent snake_case schema.


In [4]:
timetable = tt.rename(columns={
    "Train No": "train_no", "Train Name": "train_name", "SEQ": "sequence",
    "Station Code": "station_code", "Station Name": "station_name",
    "Arrival time": "arrival_time", "Departure Time": "departure_time",
    "Distance": "distance_km", "Source Station": "source_station_code",
    "Source Station Name": "source_station_name",
    "Destination Station": "destination_station_code",
    "Destination Station Name": "destination_station_name",
})[["train_no", "train_name", "sequence", "station_code", "station_name",
    "arrival_time", "departure_time", "distance_km",
    "source_station_code", "source_station_name",
    "destination_station_code", "destination_station_name"]]

timetable.to_csv(f"{OUT}/02_train_timetable.csv", index=False)
print(timetable.shape)
timetable.head()


(186119, 12)


,train_no,train_name,sequence,station_code,station_name,arrival_time,departure_time,distance_km,source_station_code,source_station_name,destination_station_code,destination_station_name
0,10103,MANDOVI EXPR,1,CSMT,CST-MUMBAI,07:10:00,07:10:00,0.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
1,10103,MANDOVI EXPR,2,DR,DADAR,07:22:00,07:25:00,8.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
2,10103,MANDOVI EXPR,3,TNA,THANE,07:46:00,07:50:00,33.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
3,10103,MANDOVI EXPR,4,PNVL,PANVEL,08:25:00,08:30:00,70.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.
4,10103,MANDOVI EXPR,5,MNI,MANGAON,10:34:00,10:35:00,177.0,CSMT,CST-MUMBAI,MAO,MADGOAN JN.


## File 3 — `03_section_traffic_derived.csv`
**Question answered: How busy is each track section, by hour?**

A "section" is the stretch of track between two consecutive stations on a train's route (e.g. NDLS→GZB). We look at every train's route, take each pair of consecutive stops, and count how many distinct trains pass through that section at each departure hour.

⚠️ **Limitation:** the raw timetable has no "which days does this train run" column, so this counts *scheduled* trains per hour, not confirmed daily-actual traffic — every train is treated as if it runs every day of the week.

We normalize direction (NDLS→GZB and GZB→NDLS become the same `section_id`) by alphabetically sorting the station-code pair.


In [5]:
tt["next_station"] = tt.groupby("Train No")["Station Code"].shift(-1)
tt["dep_hour"] = pd.to_datetime(tt["Departure Time"], format="%H:%M:%S", errors="coerce").dt.hour

hops = tt.dropna(subset=["next_station"]).copy()
hops["section_id"] = hops.apply(lambda r: "-".join(sorted([r["Station Code"], r["next_station"]])), axis=1)
hops["from_station"] = hops.apply(lambda r: sorted([r["Station Code"], r["next_station"]])[0], axis=1)
hops["to_station"] = hops.apply(lambda r: sorted([r["Station Code"], r["next_station"]])[1], axis=1)

section_hour = (hops.dropna(subset=["dep_hour"])
                .groupby(["section_id", "from_station", "to_station", "dep_hour"])
                .size().reset_index(name="trains_count"))
section_hour = section_hour.rename(columns={"dep_hour": "hour_of_day"})
section_hour["hour_of_day"] = section_hour["hour_of_day"].astype(int)

def traffic_level(n):
    if n >= 15: return "VERY_HIGH"
    if n >= 8: return "HIGH"
    if n >= 3: return "MEDIUM"
    return "LOW"

section_hour["traffic_level"] = section_hour["trains_count"].apply(traffic_level)
section_hour = section_hour.sort_values(["section_id", "hour_of_day"]).reset_index(drop=True)
section_hour.to_csv(f"{OUT}/03_section_traffic_derived.csv", index=False)

total_trains_per_section = (hops.groupby(["section_id", "from_station", "to_station"])
                             .size().reset_index(name="total_scheduled_trains")
                             .sort_values("total_scheduled_trains", ascending=False))

print(section_hour.shape, "| unique sections:", section_hour['section_id'].nunique())
total_trains_per_section.head(10)


(101130, 6) | unique sections: 14465


,section_id,from_station,to_station,total_scheduled_trains
5217,CLA-MTN,CLA,MTN,393
12001,MLND-TNA,MLND,TNA,392
3343,BNXR-DDJ,BNXR,DDJ,392
6677,DR-MTN,DR,MTN,392
3346,BNXR-SDAH,BNXR,SDAH,387
3192,BND-VK,BND,VK,360
7382,GC-VK,GC,VK,358
7383,GC-VVH,GC,VVH,356
12000,MLND-NHU,MLND,NHU,355
3190,BND-NHU,BND,NHU,355


## Choosing which real sections get maintenance/asset data

There are 14,465 unique real sections — too many to usefully plan maintenance for in a prototype. We take the **800 busiest real sections** (by scheduled train count) so the optimizer has meaningful high-traffic corridors to actually solve for.

This is the key fix that makes files 4–6 (synthetic) actually connect to file 3 (real): every synthetic record below is tagged with a `section_id` that genuinely exists in the real derived data, instead of an invented code.


In [6]:
top_sections = total_trains_per_section.head(800).reset_index(drop=True)
section_pool = top_sections["section_id"].tolist()
section_lookup = top_sections.set_index("section_id")[["from_station", "to_station", "total_scheduled_trains"]]
len(section_pool)


800

## File 4 — `04_asset_register.csv`  *(synthetic)*
**Question answered: What maintenance equipment exists?**

No public inventory of Indian Railways maintenance equipment exists, so this is simulated — but each asset is assigned a home section from the real pool above.


In [7]:
ASSET_TYPES = ["Tamping Machine", "Ballast Regulator", "Rail Crane",
               "Tower Wagon", "OHE Maintenance Special", "Dynamic Track Stabilizer"]
N_ASSETS = 400
asset_rows = []
for i in range(N_ASSETS):
    a_type = rng.choice(ASSET_TYPES)
    sec = rng.choice(section_pool)
    loc = section_lookup.loc[sec, "from_station"]
    status = rng.choice(["Available", "In Use", "Under Repair"], p=[0.70, 0.20, 0.10])
    capacity = {"Tamping Machine": "2.5 km/hr", "Ballast Regulator": "3 km/hr",
                "Rail Crane": "40 tonnes", "Tower Wagon": "6m reach",
                "OHE Maintenance Special": "N/A", "Dynamic Track Stabilizer": "1.8 km/hr"}[a_type]
    asset_rows.append(dict(
        asset_id=f"{a_type[:2].upper()}{i+1:03d}",
        asset_type=a_type,
        asset_name=f"{a_type} Unit {i+1}",
        home_section_id=sec,
        location_station=loc,
        status=status,
        capacity=capacity,
    ))

asset_register = pd.DataFrame(asset_rows)
asset_register.to_csv(f"{OUT}/04_asset_register.csv", index=False)
print(asset_register.shape)
asset_register.head()


(400, 7)


,asset_id,asset_type,asset_name,home_section_id,location_station,status,capacity
0,DY001,Dynamic Track Stabilizer,Dynamic Track Stabilizer Unit 1,KTYM-TRVL,KTYM,In Use,1.8 km/hr
1,TO002,Tower Wagon,Tower Wagon Unit 2,BHKA-KBGH,BHKA,Available,6m reach
2,TA003,Tamping Machine,Tamping Machine Unit 3,PNBE-RJPB,PNBE,In Use,2.5 km/hr
3,DY004,Dynamic Track Stabilizer,Dynamic Track Stabilizer Unit 4,BNXR-SDAH,BNXR,In Use,1.8 km/hr
4,TA005,Tamping Machine,Tamping Machine Unit 5,FA-MJ,FA,Available,2.5 km/hr


## File 5 — `05_maintenance_blocks.csv`  *(synthetic)*
**Question answered: What maintenance work is needed, where, and for how long?**

Each block is tagged with a real `section_id`. Priority is derived from that section's real traffic volume (busier real sections → higher priority). Block start hours are biased toward low-traffic night hours, mirroring real block-planning practice.


In [8]:
BLOCK_TYPES = {
    "Track Repair": ("Tamping Machine", 180), "Rail Welding": ("Rail Crane", 90),
    "Ballast Renewal": ("Ballast Regulator", 220), "OHE Maintenance": ("OHE Maintenance Special", 150),
    "Points Renewal": ("Rail Crane", 200), "Track Stabilization": ("Dynamic Track Stabilizer", 160),
    "Signal Maintenance": ("Tower Wagon", 100),
}
WEATHER = ["Clear", "Light Rain", "Heavy Rain", "Fog", "Extreme Heat"]
start_date = datetime(2023, 1, 1)
N_BLOCKS = 15000
block_rows = []
for i in range(N_BLOCKS):
    sec = rng.choice(section_pool)
    m_type = rng.choice(list(BLOCK_TYPES.keys()))
    req_asset_type, base_dur = BLOCK_TYPES[m_type]
    date = start_date + timedelta(days=int(rng.integers(0, 1095)))
    traffic = int(section_lookup.loc[sec, "total_scheduled_trains"])
    priority = "High" if traffic > 250 else ("Medium" if traffic > 100 else "Low")

    planned_duration = float(np.clip(rng.normal(base_dur, base_dur * 0.15), 30, 480))
    start_hour = int(rng.choice([0,1,2,3,4,22,23], p=[0.2,0.2,0.15,0.1,0.1,0.15,0.1]))
    planned_start = f"{start_hour:02d}:00"
    planned_end_dt = (datetime(2000,1,1,start_hour) + timedelta(minutes=planned_duration))
    planned_end = planned_end_dt.strftime("%H:%M")

    weather = rng.choice(WEATHER, p=[0.6, 0.15, 0.1, 0.1, 0.05])
    overrun = float(np.clip(rng.normal(10 + (15 if weather in ["Heavy Rain","Fog"] else 0), 20), -25, 200))
    actual_duration = max(10, planned_duration + overrun)

    block_rows.append(dict(
        block_id=f"B{i+1:05d}",
        section_id=sec,
        date=date.strftime("%Y-%m-%d"),
        maintenance_type=m_type,
        planned_start=planned_start,
        planned_end=planned_end,
        planned_duration_min=round(planned_duration,1),
        required_asset_type=req_asset_type,
        priority=priority,
        weather=weather,
        actual_duration_min=round(actual_duration,1),
        overrun_min=round(overrun,1),
        overrun_flag=int(overrun > 10),
    ))

maintenance_blocks = pd.DataFrame(block_rows).sort_values("date").reset_index(drop=True)
maintenance_blocks.to_csv(f"{OUT}/05_maintenance_blocks.csv", index=False)
print(maintenance_blocks.shape)
maintenance_blocks.head()


(15000, 13)


,block_id,section_id,date,maintenance_type,planned_start,planned_end,planned_duration_min,required_asset_type,priority,weather,actual_duration_min,overrun_min,overrun_flag
0,B09264,KOG-RIS,2023-01-01,Signal Maintenance,00:00,01:39,99.4,Tower Wagon,Medium,Extreme Heat,116.0,16.5,1
1,B08950,MS-MSB,2023-01-01,Points Renewal,00:00,02:38,158.2,Rail Crane,Low,Light Rain,195.4,37.3,1
2,B14530,UBC-UMB,2023-01-01,Rail Welding,02:00,02:54,54.4,Rail Crane,Low,Clear,84.8,30.4,1
3,B01285,ST-UDN,2023-01-01,Ballast Renewal,00:00,04:00,240.1,Ballast Regulator,Low,Light Rain,262.6,22.5,1
4,B03149,MRGA-NLDA,2023-01-01,Track Stabilization,22:00,00:08,128.2,Dynamic Track Stabilizer,Low,Light Rain,155.7,27.6,1


## File 6 — `06_daily_asset_availability.csv`  *(synthetic)*
**Question answered: Is a given asset free on a given day?**

For each asset, we sample 40 days out of a shared pool of 800 dates and mark availability (85% available, 15% unavailable with a reason).


In [9]:
dates = pd.date_range("2023-01-01", "2025-12-31", freq="D")
sample_dates = rng.choice(dates, size=800, replace=False)
avail_rows = []
for asset_id in asset_register["asset_id"]:
    for d in rng.choice(sample_dates, size=40, replace=False):
        available = rng.random() < 0.85
        reason = "" if available else rng.choice(["Scheduled Maintenance", "Breakdown", "In Use Elsewhere"])
        avail_rows.append(dict(
            date=pd.Timestamp(d).strftime("%Y-%m-%d"),
            asset_id=asset_id,
            available="YES" if available else "NO",
            available_from="00:00" if available else "",
            available_until="23:59" if available else "",
            reason_unavailable=reason,
        ))

daily_availability = pd.DataFrame(avail_rows).sort_values(["date","asset_id"]).reset_index(drop=True)
daily_availability.to_csv(f"{OUT}/06_daily_asset_availability.csv", index=False)
print(daily_availability.shape)
daily_availability.head()


(16000, 6)


,date,asset_id,available,available_from,available_until,reason_unavailable
0,2023-01-01,BA349,NO,,,In Use Elsewhere
1,2023-01-01,DY138,YES,00:00,23:59,
2,2023-01-01,DY194,YES,00:00,23:59,
3,2023-01-01,DY216,YES,00:00,23:59,
4,2023-01-01,DY328,YES,00:00,23:59,


## Sanity check — do files 3 and 5 actually join?

This confirms every `section_id` used in the maintenance table exists in the real derived traffic table.


In [10]:
shared = len(set(section_hour['section_id']) & set(maintenance_blocks['section_id']))
total = maintenance_blocks['section_id'].nunique()
print(f"Sections shared between file 3 and file 5: {shared} / {total}")
assert shared == total, "Join is broken — some maintenance sections aren't real!"
print("✅ All maintenance sections are real, derived sections. The schema is fully joinable.")


Sections shared between file 3 and file 5: 800 / 800
✅ All maintenance sections are real, derived sections. The schema is fully joinable.
